# Evaluación de modelos utilizando datos etiquetados y validación con Weights & Biases

Este cuaderno ejecuta automáticamente la evaluación del experimento activo, el cual se define en el archivo de configuración `../config/vae_experiments.json`. Su objetivo principal no es el entrenamiento de modelos, sino la ejecución de los scripts reproducibles del proyecto. Genera resultados en formato CSV y JSON dentro de la ruta `datos/analisis` y registra las métricas correspondientes en la plataforma Weights & Biases (W&B) para posibilitar comparativas sistemáticas.

Procedimiento operativo estándar:

1. Modificar el parámetro `active_experiment` en el archivo `../config/vae_experiments.json`.
2. Ejecutar secuencialmente el cuaderno `07_train_vae_experiments.ipynb`.
3. Ejecutar este cuaderno sin necesidad de modificar el contenido de sus celdas.

La variante del modelo que se evaluará es extraída del mismo parámetro `active_experiment`, asegurando la sincronización técnica entre los cuadernos 7 y 6.


In [ ]:
from pathlib import Path
import json
import os
import subprocess
import sys

import matplotlib.pyplot as plt
import pandas as pd
import numpy as np

def find_repo_root(start: Path) -> Path:
    for candidate in [start, *start.parents]:
        if (candidate / "app").is_dir() and (candidate / "tools").is_dir() and (candidate / "notebooks").is_dir():
            return candidate
    raise RuntimeError("No se encontró el directorio raíz de inference_server")

REPO_ROOT = find_repo_root(Path.cwd().resolve())
TFM_ROOT = REPO_ROOT.parent
os.chdir(REPO_ROOT)
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

print(f"REPO_ROOT={REPO_ROOT}")
print(f"TFM_ROOT={TFM_ROOT}")


## Configuración de los parámetros del experimento

Por defecto, se evaluará el mismo experimento configurado como `active_experiment` que es utilizado en el cuaderno 7. Se recomienda encarecidamente no alterar esta celda de código durante el proceso iterativo de optimización de modelos; los cambios deben realizarse exclusivamente en el archivo de configuración JSON.


In [ ]:
CONFIG_PATH = REPO_ROOT / "config" / "vae_experiments.json"
config_doc = json.loads(CONFIG_PATH.read_text(encoding="utf-8"))

# Mantener en None para utilizar config_doc["active_experiment"], de igual forma que en el cuaderno 7.
EXPERIMENT_NAME = None
EXPERIMENT_NAME = EXPERIMENT_NAME or config_doc["active_experiment"]

if EXPERIMENT_NAME not in config_doc["experiments"]:
    available = ", ".join(sorted(config_doc["experiments"]))
    raise KeyError(f"Experimento no definido: {EXPERIMENT_NAME}. Disponibles: {available}")

EXPERIMENT_CONFIG = dict(config_doc.get("defaults", {}))
EXPERIMENT_CONFIG.update(config_doc["experiments"][EXPERIMENT_NAME])
EXPERIMENT_CONFIG["experiment_name"] = EXPERIMENT_NAME
EXPERIMENT_CONFIG["config_path"] = str(CONFIG_PATH)

wandb_config = config_doc.get("wandb", {})
USE_WANDB = bool(wandb_config.get("enabled", True))
WANDB_PROJECT = wandb_config.get("project", "tfm-railway-anomaly")
WANDB_ENTITY = wandb_config.get("entity")

TRAIN_RUN_NAME = EXPERIMENT_CONFIG["run_name"]
RUN_NAME = f"eval-{TRAIN_RUN_NAME}"
MODEL_PATH = f"models/{EXPERIMENT_CONFIG['model_filename']}"

RAW_ROUTE_CSV = "../datos/brutos/real_anomaly_route_001.csv"
CONTROLLED_CSV = "../datos/brutos/real_anomaly_controlled_10laps_001.csv"
NORMAL_TRAIN_CSV = "../datos/brutos/real_movement_004.csv"

WINDOW_SIZE = int(EXPERIMENT_CONFIG["window_size"])
WINDOW_STEP = int(EXPERIMENT_CONFIG["window_step"])
RMS_PERCENTILE = float(EXPERIMENT_CONFIG.get("rms_percentile", 99.5))

ANALYSIS_DIR = TFM_ROOT / "datos" / "analysis" / RUN_NAME
FIGURE_DIR = TFM_ROOT / "datos" / "figures" / "model_evaluation" / RUN_NAME
ANALYSIS_DIR.mkdir(parents=True, exist_ok=True)
FIGURE_DIR.mkdir(parents=True, exist_ok=True)

LABELS_CSV = ANALYSIS_DIR / "route_labels_windows.csv"
IGNORE_INTERVALS_CSV = ANALYSIS_DIR / "route_ignore_intervals.csv"
ANOMALY_SEGMENTS_CSV = ANALYSIS_DIR / "route_anomaly_segments.csv"
NORMAL_SEGMENTS_CSV = ANALYSIS_DIR / "route_normal_segments.csv"
LABELED_OUTPUT = ANALYSIS_DIR / "labeled_window_predictions.csv"
LABELED_SUMMARY = ANALYSIS_DIR / "labeled_window_summary.json"
CONTROLLED_OUTPUT = ANALYSIS_DIR / "controlled_10laps_scores.csv"
CONTROLLED_SUMMARY = ANALYSIS_DIR / "controlled_10laps_summary.json"

print(f"CONFIG_PATH={CONFIG_PATH}")
print(f"EXPERIMENT_NAME={EXPERIMENT_NAME}")
print(f"TRAIN_RUN_NAME={TRAIN_RUN_NAME}")
print(f"EVAL_RUN_NAME={RUN_NAME}")
print(f"MODEL_PATH={MODEL_PATH}")
print(f"WINDOW_SIZE={WINDOW_SIZE} WINDOW_STEP={WINDOW_STEP}")
print(json.dumps(EXPERIMENT_CONFIG, indent=2))

for path in [MODEL_PATH, RAW_ROUTE_CSV, CONTROLLED_CSV, NORMAL_TRAIN_CSV]:
    resolved = (REPO_ROOT / path).resolve()
    print(f"{path}: {'OK' if resolved.exists() else 'MISSING'} -> {resolved}")
print(f"Etiquetas de las capturas experimentales: {LABELS_CSV}")


## Evaluación principal contrastando contra ventanas etiquetadas

Esta representa la fase de evaluación más crítica para el refinamiento iterativo de los modelos: se establece una comparativa de los scores VAE y los umbrales RMS frente a las categorías `normal_quiet`, `anomaly_impact` y `anomaly_slip_candidate`, excluyendo explícitamente las ventanas catalogadas como `ignore`.

In [ ]:
def run_command(command: list[str]) -> None:
    print(" ".join(command))
    completed = subprocess.run(command, cwd=REPO_ROOT, text=True, capture_output=True)
    if completed.stdout:
        print(completed.stdout)
    if completed.stderr:
        print(completed.stderr)
    completed.check_returncode()


## Generación de etiquetas específicas ajustadas al experimento

Se procede a regenerar las etiquetas en el directorio `datos/analisis/<RUN_NAME>/` aplicando los valores de `WINDOW_SIZE` y `WINDOW_STEP` asociados al modelo que se está evaluando. Esto previene de forma sistemática la combinación inconsistente de modelos que operan a 2 s o 3 s de ventana con conjuntos de etiquetas definidos para 1 s.


In [ ]:
label_cmd = [
    sys.executable,
    "herramientas/build_labeled_route_dataset.py",
    "--raw-csv", str((REPO_ROOT / RAW_ROUTE_CSV).resolve()),
    "--windows-csv", str(LABELS_CSV),
    "--ignore-intervals-csv", str(IGNORE_INTERVALS_CSV),
    "--anomaly-segments-csv", str(ANOMALY_SEGMENTS_CSV),
    "--normal-segments-csv", str(NORMAL_SEGMENTS_CSV),
    "--window-size", str(WINDOW_SIZE),
    "--window-step", str(WINDOW_STEP),
]
run_command(label_cmd)

labels_preview = pd.read_csv(LABELS_CSV)
ignore_preview = pd.read_csv(IGNORE_INTERVALS_CSV)
display(labels_preview["label"].value_counts().rename_axis("label").reset_index(name="windows"))
display(ignore_preview)


In [ ]:
labeled_cmd = [
    sys.executable,
    "herramientas/evaluate_labeled_windows.py",
    "--raw-csv", RAW_ROUTE_CSV,
    "--labels-csv", str(LABELS_CSV),
    "--normal-train-csv", NORMAL_TRAIN_CSV,
    "--vae-model", MODEL_PATH,
    "--window-size", str(WINDOW_SIZE),
    "--window-step", str(WINDOW_STEP),
    "--rms-percentile", str(RMS_PERCENTILE),
    "--output", str(LABELED_OUTPUT),
    "--summary-output", str(LABELED_SUMMARY),
    "--wandb-project", WANDB_PROJECT,
    "--wandb-entity", WANDB_ENTITY or "",
    "--wandb-run-name", f"{RUN_NAME}-labeled",
]
if USE_WANDB:
    labeled_cmd.append("--wandb")

run_command(labeled_cmd)


In [ ]:
with open(LABELED_SUMMARY, "r", encoding="utf-8") as file:
    labeled_summary = json.load(file)

metrics = labeled_summary["metrics"]
details = labeled_summary["details"]
main_metrics = pd.DataFrame([
    {
        "detector": "VAE actual threshold",
        "precision": metrics["vae_precision"],
        "recall": metrics["vae_recall"],
        "f1": metrics["vae_f1"],
        "accuracy": metrics["vae_accuracy"],
        "recall_impact": metrics.get("vae_recall_anomaly_impact"),
        "recall_slip": metrics.get("vae_recall_anomaly_slip_candidate"),
        "fp": details["vae"]["false_positive"],
        "fn": details["vae"]["false_negative"],
    },
    {
        "detector": "RMS baseline",
        "precision": metrics["rms_precision"],
        "recall": metrics["rms_recall"],
        "f1": metrics["rms_f1"],
        "accuracy": metrics["rms_accuracy"],
        "recall_impact": metrics.get("rms_recall_anomaly_impact"),
        "recall_slip": metrics.get("rms_recall_anomaly_slip_candidate"),
        "fp": details["rms"]["false_positive"],
        "fn": details["rms"]["false_negative"],
    },
    {
        "detector": "VAE best threshold",
        "precision": details["vae_best"]["precision"],
        "recall": details["vae_best"]["recall"],
        "f1": details["vae_best"]["f1"],
        "accuracy": details["vae_best"]["accuracy"],
        "recall_impact": None,
        "recall_slip": None,
        "fp": details["vae_best"]["false_positive"],
        "fn": details["vae_best"]["false_negative"],
    },
])

display(main_metrics.style.format(precision=3))
print(f"VAE best threshold normalizado: {metrics['vae_best_threshold']:.6f}")
print(f"RMS best threshold: {metrics['rms_best_threshold']:.6f}")


## Distribución de los scores en función de la etiqueta de validación

Este análisis gráfico facilita la determinación visual sobre si una variante particular del modelo es capaz de distinguir con mayor eficacia anomalías mecánicas sutiles. El escenario ideal se materializa cuando las muestras de la categoría `anomaly_slip_candidate` presentan una desviación significativa hacia scores más elevados, sin que esto repercuta en un incremento análogo de los scores en la categoría `normal_quiet`.

In [ ]:
predictions = pd.read_csv(LABELED_OUTPUT)

fig, axes = plt.subplots(1, 2, figsize=(14, 5), sharey=False)
predictions.boxplot(column="vae_score_normalized", by="label", ax=axes[0], grid=False)
axes[0].set_title("VAE Score normalizado según etiqueta")
axes[0].set_xlabel("Etiqueta")
axes[0].set_ylabel("Score normalizado")
axes[0].tick_params(axis="x", rotation=25)
axes[0].axhline(1.0, color="red", linestyle="--", linewidth=1, label="Umbral de detección actual")
axes[0].legend()

predictions.boxplot(column="rms_score", by="label", ax=axes[1], grid=False)
axes[1].set_title("RMS Score según etiqueta")
axes[1].set_xlabel("Etiqueta")
axes[1].set_ylabel("RMS energy")
axes[1].tick_params(axis="x", rotation=25)
axes[1].axhline(predictions["rms_threshold"].iloc[0], color="red", linestyle="--", linewidth=1, label="Umbral RMS")
axes[1].legend()

plt.suptitle("")
plt.tight_layout()
fig_path = FIGURE_DIR / "score_distribution_by_label.png"
plt.savefig(fig_path, dpi=160)
print(f"Gráfica exportada exitosamente: {fig_path}")
plt.show()


## Matrices de confusión y análisis paramétrico de umbrales (Threshold Sweep)

Las tablas resultantes constituyen las herramientas más idóneas para establecer comparativas de rendimiento entre distintos modelos. Permiten evaluar la tasa de falsos positivos en ventanas de comportamiento normal, el valor de sensibilidad (recall) segregado por clase y la dinámica del modelo frente a la variación sistemática del umbral de decisión (threshold).


In [ ]:
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay, precision_recall_fscore_support

fig, axes = plt.subplots(1, 2, figsize=(11, 4))
for ax, pred_col, title in [(axes[0], "vae_prediction", "VAE"), (axes[1], "rms_prediction", "RMS")]:
    cm = confusion_matrix(predictions["binary_label"], predictions[pred_col], labels=["normal", "anomaly"])
    disp = ConfusionMatrixDisplay(cm, display_labels=["normal", "anomaly"])
    disp.plot(ax=ax, cmap="Blues", colorbar=False)
    ax.set_title(title)
plt.tight_layout()
fig_path = FIGURE_DIR / "confusion_matrices_binary.png"
plt.savefig(fig_path, dpi=160)
print(f"Gráfica exportada exitosamente: {fig_path}")
plt.show()

threshold_rows = []
for threshold in [0.70, 0.75, 0.80, 0.85, 0.90, 0.95, 1.00, 1.10, 1.20]:
    pred = np.where(predictions["vae_score_normalized"] >= threshold, "anomaly", "normal")
    precision, recall, f1, _ = precision_recall_fscore_support(
        predictions["binary_label"], pred, labels=["anomaly"], average="binary", pos_label="anomaly", zero_division=0
    )
    fp = int(((predictions["binary_label"] == "normal") & (pred == "anomaly")).sum())
    impact = predictions[predictions["label"] == "anomaly_impact"]
    slip = predictions[predictions["label"] == "anomaly_slip_candidate"]
    threshold_rows.append({
        "threshold": threshold,
        "precision": precision,
        "recall": recall,
        "f1": f1,
        "false_positive": fp,
        "recall_impact": float((impact["vae_score_normalized"] >= threshold).mean()) if len(impact) else np.nan,
        "recall_slip": float((slip["vae_score_normalized"] >= threshold).mean()) if len(slip) else np.nan,
    })
threshold_df = pd.DataFrame(threshold_rows)
display(threshold_df.style.format(precision=3))
threshold_df.to_csv(ANALYSIS_DIR / "threshold_sweep.csv", index=False)

fig, ax = plt.subplots(figsize=(9, 5))
ax.plot(threshold_df["threshold"], threshold_df["recall_impact"], marker="o", label="recall impact")
ax.plot(threshold_df["threshold"], threshold_df["recall_slip"], marker="o", label="recall slip")
ax.plot(threshold_df["threshold"], threshold_df["false_positive"], marker="o", label="false positives")
ax.set_xlabel("Score VAE normalizado (Threshold)")
ax.set_title("Análisis paramétrico de variación de umbral VAE (Threshold Sweep)")
ax.legend()
plt.tight_layout()
fig_path = FIGURE_DIR / "threshold_sweep.png"
plt.savefig(fig_path, dpi=160)
print(f"Gráfica exportada exitosamente: {fig_path}")
plt.show()


## Prueba empírica en entorno controlado (10 Vueltas a la vía)

Esta validación no aplica un etiquetado exhaustivo a nivel de ventana. Su propósito principal es verificar si se identifican aproximadamente 20 eventos anómalos a lo largo de 10 recorridos completos del tren. Actúa como un test de verificación rápida independiente (sanity check) para mitigar el riesgo de sobreajuste (overfitting) respecto a los datos etiquetados.

In [ ]:
controlled_cmd = [
    sys.executable,
    "herramientas/evaluate_controlled_run.py",
    "--raw-csv", CONTROLLED_CSV,
    "--normal-train-csv", NORMAL_TRAIN_CSV,
    "--vae-model", MODEL_PATH,
    "--window-size", str(WINDOW_SIZE),
    "--window-step", str(WINDOW_STEP),
    "--output", str(CONTROLLED_OUTPUT),
    "--summary-output", str(CONTROLLED_SUMMARY),
    "--wandb-project", WANDB_PROJECT,
    "--wandb-entity", WANDB_ENTITY or "",
    "--wandb-run-name", f"{RUN_NAME}-controlled",
]
if USE_WANDB:
    controlled_cmd.append("--wandb")

run_command(controlled_cmd)


In [ ]:
with open(CONTROLLED_SUMMARY, "r", encoding="utf-8") as file:
    controlled_summary = json.load(file)

thresholds = pd.DataFrame(controlled_summary["thresholds"])
display(thresholds[["detector", "threshold", "event_count", "expected_events", "event_count_error"]])

scores = pd.read_csv(CONTROLLED_OUTPUT)
fig, ax = plt.subplots(figsize=(14, 4))
ax.plot(scores["relative_center_s"], scores["vae_score_normalized"], label="VAE Score normalizado")
ax.axhline(1.0, color="red", linestyle="--", linewidth=1, label="Umbral actual de evaluación")
ax.axhline(0.85, color="orange", linestyle="--", linewidth=1, label="Umbral candidato recomendado")
ax.set_title("Ejecución controlada (10 Vueltas): Evolución del VAE Score por ventana")
ax.set_xlabel("Tiempo relativo de ejecución (s)")
ax.set_ylabel("score normalizado")
ax.legend()
plt.tight_layout()
fig_path = FIGURE_DIR / "controlled_10laps_vae_score.png"
plt.savefig(fig_path, dpi=160)
print(f"Gráfica exportada exitosamente: {fig_path}")
plt.show()

## Resumen analítico comparativo final

Esta matriz constituye la referencia ejecutiva para evaluar de manera objetiva si el experimento actual aporta mejoras medibles con respecto a versiones anteriores. Las familias de métricas presentes en la tabla son análogas a las que se consolidan en la plataforma Weights & Biases bajo los prefijos organizativos `eval/`, `label_recall/` y `controlled/`.


In [ ]:
controlled_thresholds = pd.DataFrame(controlled_summary["thresholds"])
vae_100 = controlled_thresholds[(controlled_thresholds["detector"] == "vae") & (controlled_thresholds["threshold"].round(2) == 1.00)]
vae_085 = controlled_thresholds[(controlled_thresholds["detector"] == "vae") & (controlled_thresholds["threshold"].round(2) == 0.85)]
rms_events = controlled_thresholds[controlled_thresholds["detector"] == "rms"]

final_summary = pd.DataFrame([
    {
        "experiment": EXPERIMENT_NAME,
        "model": MODEL_PATH,
        "window_size": WINDOW_SIZE,
        "window_step": WINDOW_STEP,
        "vae_f1": metrics["vae_f1"],
        "vae_precision": metrics["vae_precision"],
        "vae_recall": metrics["vae_recall"],
        "vae_fp": details["vae"]["false_positive"],
        "vae_fn": details["vae"]["false_negative"],
        "impact_recall": metrics.get("vae_recall_anomaly_impact"),
        "slip_recall": metrics.get("vae_recall_anomaly_slip_candidate"),
        "vae_best_threshold": metrics["vae_best_threshold"],
        "vae_best_f1": metrics["vae_best_f1"],
        "controlled_events_at_1_00": int(vae_100["event_count"].iloc[0]) if len(vae_100) else None,
        "controlled_error_at_1_00": int(vae_100["event_count_error"].iloc[0]) if len(vae_100) else None,
        "controlled_events_at_0_85": int(vae_085["event_count"].iloc[0]) if len(vae_085) else None,
        "controlled_error_at_0_85": int(vae_085["event_count_error"].iloc[0]) if len(vae_085) else None,
        "rms_f1": metrics["rms_f1"],
        "rms_slip_recall": metrics.get("rms_recall_anomaly_slip_candidate"),
        "rms_controlled_events": int(rms_events["event_count"].iloc[0]) if len(rms_events) else None,
    }
])
summary_csv = ANALYSIS_DIR / "final_comparable_summary.csv"
final_summary.to_csv(summary_csv, index=False)
display(final_summary.T.rename(columns={0: "value"}))
print(f"El resumen analítico comparativo ha sido exportado a: {summary_csv}")


## Criterios de toma de decisiones para la siguiente iteración de modelado

- Mantener un nivel de sensibilidad (`recall_impact`) asintótico a 1.0 para los eventos críticos.
- Incrementar de forma paulatina la sensibilidad frente a derrapes (`recall_slip`) sin provocar una degradación en la tasa de falsos positivos sobre segmentos nominales (`normal_quiet`).
- Verificar que, de forma constante, la ejecución empírica controlada registre una cantidad cercana a 20 eventos mecánicos.
- Si el umbral óptimo decae significativamente y ello conlleva la inclusión de falsos positivos sistémicos, será imperativo perfeccionar la ingeniería de características (feature engineering) o la arquitectura estructural del modelo, en lugar de limitarse a ajustes de hiperparámetros de decisión (threshold tuning).

Posibles vías de optimización para iteraciones futuras:

- Entrenamiento de un modelo VAE ampliando la ventana temporal a 2 s o 3 s.
- Ampliación del vector de entrada incorporando características derivadas, tales como: magnitud del vector aceleración, derivadas temporales por eje, o filtros de media móvil/diferencias finitas.
- Desarrollo de un modelo de referencia (baseline) compuesto por variables estadísticas: RMS de vibración por eje + aceleración máxima de pico + desviación de la media del vector longitudinal.
- Arquitectura híbrida: Combinación matemática ponderada del VAE score y métricas físicas deterministas para una cobertura integral de fenómenos de patinaje y frenada brusca.
